In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

### **Create Table to store what data we have loaded in**
---

In [0]:
%sql
DROP TABLE IF EXISTS test01.bronze.control;

CREATE TABLE test01.bronze.control (
    id INT,
    path STRING,
    table STRING
)

#### Check a file in path
---

In [0]:
path = '/Volumes/test01/bronze/source'

file_to_table = {
    "cust_info.csv": "crm_cust_info",
    "prd_info.csv": "crm_prd_info",
    "sales_details.csv": "crm_sales_details",
    "CUST_AZ12.csv": "erp_cust_az12",
    "LOC_A101.csv": "erp_loc_a101",
    "PX_CAT_G1V2.csv": "erp_px_cat_g1v2"
}

def list_all_files(path):
    files = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            files.extend(list_all_files(item.path))
        else:
            if item.path.lower().endswith(".csv"):
                files.append(item.path)
    return files

all_csv_path = list_all_files(path)

all_files = []

for files_path in all_csv_path:
    file_name = files_path.split('/')[-1]

    if file_name in file_to_table:
        all_files.append({
            'path': files_path,
            'table': file_to_table[file_name]
        })
    else:
        print (f'Warning: No mapping for {file_name}')
for f in all_files:
    print (f)

{'path': 'dbfs:/Volumes/test01/bronze/source/crm/cust_info.csv', 'table': 'crm_cust_info'}
{'path': 'dbfs:/Volumes/test01/bronze/source/crm/prd_info.csv', 'table': 'crm_prd_info'}
{'path': 'dbfs:/Volumes/test01/bronze/source/crm/sales_details.csv', 'table': 'crm_sales_details'}
{'path': 'dbfs:/Volumes/test01/bronze/source/erp/CUST_AZ12.csv', 'table': 'erp_cust_az12'}
{'path': 'dbfs:/Volumes/test01/bronze/source/erp/LOC_A101.csv', 'table': 'erp_loc_a101'}
{'path': 'dbfs:/Volumes/test01/bronze/source/erp/PX_CAT_G1V2.csv', 'table': 'erp_px_cat_g1v2'}
{'path': 'dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/cust_info.csv', 'table': 'crm_cust_info'}
{'path': 'dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/prd_info.csv', 'table': 'crm_prd_info'}
{'path': 'dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/sales_details.csv', 'table': 'crm_sales_details'}
{'path': 'dbfs:/Volumes/test01/bronze/source/incremental/batch_01/erp/CUST_AZ12.csv', 'table': 'erp_cust

#### Write the path for a files that we have already loaded
---

In [0]:
%sql
INSERT INTO test01.bronze.control (id, path, table)
VALUES (1, 'dbfs:/Volumes/test01/bronze/source/crm/cust_info.csv', 'crm_cust_info'),
(2, 'dbfs:/Volumes/test01/bronze/source/crm/prd_info.csv', 'crm_prd_info'),
(3, 'dbfs:/Volumes/test01/bronze/source/crm/sales_details.csv', 'crm_sales_details'),
(4, 'dbfs:/Volumes/test01/bronze/source/erp/CUST_AZ12.csv', 'erp_cust_az12'),
(5, 'dbfs:/Volumes/test01/bronze/source/erp/LOC_A101.csv', 'erp_loc_a101'),
(6, 'dbfs:/Volumes/test01/bronze/source/erp/PX_CAT_G1V2.csv', 'erp_px_cat_g1v2')

num_affected_rows,num_inserted_rows
6,6


### **Start append data into bronze layer table**
---

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Load ingested paths and normalize them
try:
    ingested_df = spark.table("test01.bronze.control").select("path")
    ingested_paths = set(
        row.path.strip().rstrip("/") for row in ingested_df.collect()
    )
except:
    ingested_paths = set()

print(f"Already ingested count: {len(ingested_paths)}")

# Loop
for item in all_files:
    file_path = item["path"].strip().rstrip("/")
    table_name = item["table"]   

    if file_path in ingested_paths:
        print(f"Already ingested → {file_path}")
    else:
        print(f"Ingesting → {file_path}")

        (
            spark.read.format("csv")
            .option("header", True)
            .load(item["path"])               # use original path for reading
            .withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", col("_metadata.file_path"))
            .write.format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(f"test01.bronze.{table_name}")
        )

        # Insert with the same normalized path
        spark.sql(f"""
            INSERT INTO test01.bronze.control (path, table)
            VALUES ('{file_path}', '{table_name}')
        """)

        print(f"Completed → {file_path}")
        print("-" * 50)

Already ingested count: 6
Already ingested → dbfs:/Volumes/test01/bronze/source/crm/cust_info.csv
Already ingested → dbfs:/Volumes/test01/bronze/source/crm/prd_info.csv
Already ingested → dbfs:/Volumes/test01/bronze/source/crm/sales_details.csv
Already ingested → dbfs:/Volumes/test01/bronze/source/erp/CUST_AZ12.csv
Already ingested → dbfs:/Volumes/test01/bronze/source/erp/LOC_A101.csv
Already ingested → dbfs:/Volumes/test01/bronze/source/erp/PX_CAT_G1V2.csv
Ingesting → dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/cust_info.csv
Completed → dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/cust_info.csv
--------------------------------------------------
Ingesting → dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/prd_info.csv
Completed → dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/prd_info.csv
--------------------------------------------------
Ingesting → dbfs:/Volumes/test01/bronze/source/incremental/batch_01/crm/sales_details.csv
Comp